In [ ]:
# =============================================
# IMPORT
# =============================================
from utility.logger         import aiLogger
from keras.callbacks        import EarlyStopping, ReduceLROnPlateau
from model.componentModel   import mainModel
from model.tepSystemModel   import SystemModel
from preProcessingDataTEP   import PreProcessData
from prettytable            import PrettyTable

from keras.optimizers       import Adam
from utility.matpilotDraw   import MatPlot
from sklearn.preprocessing  import MinMaxScaler

import tensorflow           as tf
import pandas               as pd
import matplotlib.pyplot    as plt
import numpy                as np
import os, warnings
import random
import json


In [ ]:
# ---------------------------
# Loading config file
# ---------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    configData = json.load(configFile)
config = configData["tep"]  

# ---------------------------
# Set debug level and 
# ---------------------------
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

outputDir               = os.path.expanduser(config["dataDir"])
totalTraject            = 100
normalizeHI             = True
showRedundantShutdown   = False

#### Preprocessing data

In [ ]:
# ==============================
# Config and data preprocessing
# ==============================
preData             = PreProcessData(config=config)
rawData             = preData.loadDataFromH5Format(dir              = outputDir, 
                                                   name             = "data/teps.h5",
                                                   case             = "all", 
                                                   reduceSampleRate = True, 
                                                   removeHigherThr  = False)

withRULData         = preData.addRULAndEnvironment(rawData)
# withoutInitRawData  = preData.removeInitState(inData= withRULData)



# ---------------------------
# For data
cleanData           = preData.cleanData(rawSignal           =   withRULData,
                                        columnName          =   preData.allWithoutLabel,
                                        getMovingAverage    =   False,
                                        flipToDown          =   False)
# ---------------------------
# For label, flip only
cleanData           = preData.cleanData(rawSignal           =   cleanData,
                                        columnName          =   preData.degraLiq,
                                        getMovingAverage    =   False,
                                        flipToDown          =   False)
norData             = preData.normalizeDataEachModes(rawData    = cleanData,
                                                     headline   =   preData.headline["allWithoutLabelNoBias"] + ["rul"])    # Normalize the data



#### Declare the model

In [ ]:
config["component"] = "Sys"
vaeSys = SystemModel(config = config)
vaeSys.build(input_shape=[(None, config["seqLength"], config["inReac"]), (None, 1, config["inReac"]), (None, 1)])
vaeSys.summary()


# modelDirComp1 = outputDir + "checkpoint/" + "bestModelReacNewData"
# modelDirComp2 = outputDir + "checkpoint/" + "bestModelSepNewData"
# modelDirComp3 = outputDir + "checkpoint/" + "bestModelStripNewData"
# modelDirSys   = outputDir + "checkpoint/" + "bestModelSysNewData"


modelDirComp1 = outputDir + "checkpoint/" + "bestModel34Reac"
modelDirComp2 = outputDir + "checkpoint/" + "bestModel35Sep"
modelDirComp3 = outputDir + "checkpoint/" + "bestModel36Strip"
modelDirSys   = outputDir + "checkpoint/" + "bestModel38Sys"
vaeSys.load_weights(modelDirComp1, modelDirComp2, modelDirComp3, modelDirSys)

### MAKS EACH COMPONENT IN EACH ENVIRONMENT

#### Reactor

In [ ]:
binaryMaskReac, probsReac     = vaeSys.maskCr1.getMask()
for idxEnv in range(3):
    ssTable                 = PrettyTable()
    ssTable.field_names     = ["No","sensorName","probability", "binary"]
    print(f"The number of selected sensor in the environment {idxEnv+1} is {sum(binaryMaskReac[idxEnv])}")
    for idx,idxSS in enumerate(preData.headline["allWithoutLabelNoBias"]):
        ssTable.add_row([idx,idxSS,probsReac[idxEnv][idx].numpy(),binaryMaskReac[idxEnv][idx].numpy()])

    print(ssTable)

#### Separator

In [ ]:
binaryMaskReac, probsReac     = vaeSys.maskCr2.getMask()
for idxEnv in range(3):
    ssTable                 = PrettyTable()
    ssTable.field_names     = ["No","sensorName","probability", "binary"]
    print(f"The number of selected sensor in the environment {idxEnv+1} is {sum(binaryMaskReac[idxEnv])}")
    for idx,idxSS in enumerate(preData.headline["allWithoutLabelNoBias"]):
        ssTable.add_row([idx,idxSS,probsReac[idxEnv][idx].numpy(),binaryMaskReac[idxEnv][idx].numpy()])

    print(ssTable)



#### Stripper

In [ ]:
binaryMaskReac, probsReac     = vaeSys.maskCv2.getMask()
for idxEnv in range(3):
    ssTable                 = PrettyTable()
    ssTable.field_names     = ["No","sensorName","probability", "binary"]
    print(f"The number of selected sensor in the environment {idxEnv+1} is {sum(binaryMaskReac[idxEnv])}")
    for idx,idxSS in enumerate(preData.headline["allWithoutLabelNoBias"]):
        ssTable.add_row([idx,idxSS,probsReac[idxEnv][idx].numpy(),binaryMaskReac[idxEnv][idx].numpy()])

    print(ssTable)

### HI FOR EACH ENVIRONMENT

In [ ]:
# Plot functions
import plotly.graph_objects as go
import os
def draw2DMultipleY(
                    xCoor       = [0,10],
                    yCoor       = [[0,10],[10,1]],
                    yLabel      = "Y Value",
                    xLabel      = "Time Unit",
                    title       = "XYAxis",
                    saveFig     = False,
                    direction   = "checkpoint/",
                    savedName   = "default",
                    useLegend   = True,
                    legend      = ["line1","line2"],
                    color       = ["blue","green"],
                    yLim       = [],):
    
    plt.figure()
    for idx,yValue in enumerate(yCoor):
        plt.plot(xCoor,yValue, color=color[idx])

    plt.title(title)
    # plt.xlabel(xLabel)
    # plt.ylabel(yLabel)

    if len(yLim) != 0:
        plt.ylim(yLim)
    if useLegend:
        plt.legend(legend, loc='upper left')
    plt.grid(True)
    if saveFig:
        
        if not os.path.exists(direction):
            os.makedirs(direction)
        plt.savefig(direction + savedName + '.png')
        plt.close()
    else:
        plt.show()


def draw2DMultipleY_plotly(
        xCoor=[0, 10],
        yCoor=[[0, 10], [10, 1]],
        yLabel="Y Value",
        xLabel="Time Unit",
        title="XYAxis",
        saveFig=False,
        direction="checkpoint/",
        savedName="default",
        useLegend=True,
        legend=["line1", "line2"],
        color=["blue", "green"],
        yLim=[]
    ):

    fig = go.Figure()

    for idx, yValue in enumerate(yCoor):
        fig.add_trace(go.Scatter(
            x=xCoor,
            y=yValue,
            mode='lines',
            name=legend[idx] if useLegend else None,
            line=dict(color=color[idx])
        ))

    # fig.update_layout(
    #     title=title,
    #     xaxis_title=xLabel,
    #     yaxis_title=yLabel,
    #     yaxis=dict(range=yLim if yLim else None),
    #     showlegend=useLegend
    # )
    fig.update_layout(
    title=title,
    xaxis_title=xLabel,
    yaxis_title=yLabel,
    yaxis=dict(range=yLim if yLim else None),
    showlegend=useLegend,
    plot_bgcolor='white',
    paper_bgcolor='white'
)


    if saveFig:
        if not os.path.exists(direction):
            os.makedirs(direction)
        fig.write_image(f"{direction}{savedName}.png")
    else:
        fig.show()


#### Environment 4

In [ ]:
dfHIAll = pd.DataFrame(columns=[
                                'oriReacHI',
                                'oriSepHI',
                                'oriStripHI',
                                'oriSysHI',
                                'time'
                            ])

environmentIdx      = 4
data                = norData[environmentIdx]
trajectID           = data["Id"]
reactor             = data[preData.headline["reactor"]]
separator           = data[preData.headline["separator"]]
stripper            = data[preData.headline["stripper"]]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []
hiEnv1["Sys"]       = []
rulEnv1["Sys"]     = []
hiEnv1["multipliedMatrix"]       = []
timeDraw            = []
hiEnv1["timeDraw"] = []

# timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
# trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(totalTraject):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
    # seqXTrainEnv, seqXAllSensorTrain,seqTime     = preData.getTensorData(   dataList    = [env,allSensor,timeTraj],
    #                                                                         trajectId   = trajectID,
    #                                                                         getAtIdx    = getAtIdx,   
    #                                                                         seqLength   = config["seqLength"],
    #                                                                         overlap     = config["overLap"]
    #                                                                     )
    seqXAllSensorTrain,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                                trajectId   = trajectID,  
                                                                shuffle     = False,      
                                                                getAtIdx    = getAtIdx,
                                                                seqLength   = preData.config["seqLength"],
                                                                overlap     = preData.config["overLap"]         
                                                                )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqXAllSensorTrain, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]  

    hiReac, hiSep,  hiStrip, hiSystem, predictedRul, hiMatrixOutput    = vaeSys.getHIandRUL(input = xPredict)
    # squeezedHiSystem                    = tf.reduce_mean(hiSystem,axis=-1)
    # print(squeezedHiSystem.shape)
    squeezedHiReac                                 = tf.squeeze(hiReac, axis=-1)
    squeezedHiSep                                  = tf.squeeze(hiSep, axis=-1)
    squeezedHiStrip                                = tf.squeeze(hiStrip, axis=-1)
    squeezedHiSys                                  = tf.squeeze(hiSystem, axis=-1)
    timeDraw.append(seqTimeTrain)
    print("------------------------------",squeezedHiReac)

    dfTemp      = pd.DataFrame({'id'           : np.ones(squeezedHiReac.shape)*idx,
                                'oriReacHI'    : squeezedHiReac.numpy(), 
                                'oriSepHI'     : squeezedHiSep.numpy(),
                                'oriStripHI'   : squeezedHiStrip.numpy(),
                                'oriSysHI'     : squeezedHiSys.numpy(),
                                'time'      : np.squeeze(seqTimeTrain)})
    

    dfHIAll     = pd.concat([dfHIAll,dfTemp],ignore_index=True)

dfHIAll["reacHI"]   = dfHIAll["oriReacHI"].copy()
dfHIAll["sepHI"]    = dfHIAll["oriSepHI"].copy()
dfHIAll["stripHI"]  = dfHIAll["oriStripHI"].copy()



    # hiEnv1["Reac"].append(hiReac)
    # hiEnv1["Sep"].append(hiSep)
    # hiEnv1["Strip"].append(hiStrip)
    # hiEnv1["Sys"].append(hiSystem)
    # hiEnv1["timeDraw"].append(seqTimeTrain)
    # hiEnv1["multipliedMatrix"].append(hiMatrixOutput)
    # rulEnv1["Sys"].append(predictedRul)
    # rulEnv1["Reac"].append(squeezedRulReac)
    # rulEnv1["Sep"].append(squeezedRulSep)
    # rulEnv1["Strip"].append(squeezedRulStrip)

    # break
    


# if normalizeHI:
#     [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], timeDraw , rulEnv1["Sys"], hiEnv1["multipliedMatrix"]= preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], time= timeDraw, rul = rulEnv1["Sys"], multipliedMt = hiEnv1["multipliedMatrix"], indexToShutdown = 2, showRedundantShutdown = showRedundantShutdown)
# hiEnv1["Sys"]                                  = preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], indexToShutdown = 2)


In [ ]:
print(dfHIAll)

#### Normalize for specific sample

In [ ]:
drawId = 1

hiColumns = ["reacHI", "sepHI", "stripHI"]

# ===================================
# Moving average for HIs
# -----------------------------------
print(dfHIAll[hiColumns].shape)
for _ in range(2):
    for col in hiColumns:
        dfHIAll[col] = (dfHIAll[col].transform(lambda x: x.rolling(window=15, min_periods=1).mean()))

dfHIAll             = dfHIAll[dfHIAll["time"] >= 40]               # 
dfHIAll["time"]     = dfHIAll["time"] - dfHIAll["time"].min()     # Normalize time to start from 0

# ====================================
# Normalize for HIs
# ------------------------------------
globalMax           = dfHIAll.groupby('id')[hiColumns].max().max(axis = 1).reset_index(name='globalMax')   # Get the global maximum for each ids
maxPerIdEachCol     = dfHIAll.groupby('id')[hiColumns].max().reset_index()                                 # Get the maximum value for each column across all ids

shiftMatrix         = pd.DataFrame({    'reacHI': globalMax['globalMax'] -  maxPerIdEachCol['reacHI'],
                                        'sepHI':  globalMax['globalMax'] -  maxPerIdEachCol['sepHI'],
                                        'stripHI': globalMax['globalMax'] - maxPerIdEachCol['stripHI'],
                                    })                                                                             # Create a shift matrix to adjust the HI values


dfHIAll[hiColumns]  =  dfHIAll.groupby('id')[hiColumns].apply(lambda group: group + shiftMatrix.loc[group.name,:])  # Adjust the HI values by adding the shift matrix


upperHIs            = max(globalMax['globalMax'])                                                           # Use the global maximum for normalization
hiThreshold         = max(dfHIAll.groupby('id')['stripHI'].min())


dfHIAll             = dfHIAll[dfHIAll['stripHI'] > hiThreshold]                                             # Filter out engines with HI > threshold

dfHIAll[hiColumns]  = (dfHIAll[hiColumns] - hiThreshold) / (upperHIs - hiThreshold)           # Normalize HI


dfHIAll["sysHI"]    = dfHIAll[hiColumns].min(axis=1)  
print(dfHIAll.head(5))

dfHIAll.to_excel("allHIs.xlsx", index=False)


label = pd.read_excel("~/Desktop/Dropbox/1.codeData/tepDataWithoutGit/data/labelToDraw.xlsx")
stripper_max_abs = label["Stripper"].abs().max()
data_normalized = label.copy()
data_normalized["Stripper"] = 1 + label["Stripper"] / stripper_max_abs
data_normalized["Separator"] = 1 + label["Separator"] / stripper_max_abs
data_normalized["Reactor"] = 1 + label["Reactor"] / stripper_max_abs

systemLabel     = data_normalized[["Stripper", "Separator", "Reactor"]].min(axis=1)
systemLabel     = systemLabel.iloc[1:-1]

plot2 = MatPlot()
drawData = dfHIAll[dfHIAll["id"] == 1]  # Filter for a specific engine ID, e.g., 1
print(drawData["time"])

plot2.draw2DMultipleY(xCoor =   drawData["time"], 
                      yCoor =   [drawData["reacHI"], drawData["sepHI"], drawData["stripHI"], drawData["sysHI"], systemLabel],
                      color = ["green","blue","orange","red", 'black'],
                      useLegend= False,
                      title = "Proposed method with expert knowledge",
                      xLabel = "Hours",
                      yLabel= "Health Indicator",)



In [ ]:
def subPlotMultiImgPandas(
                        inData      = None,
                        inYLabel    = ["reacHI","sepHI", "stripHI", "sysHI"],
                        subPlotName = "Name",
                        numRow      = 2,
                        numCol      = 2,
                        color       = None,
                        xLim        = [],
                        yLim        = []
                        ):
        inX         = inData["time"]
        idxTraject  = inData["id"]
        his         = inData[inYLabel]

        # print(idxTraject.unique())
        # print(max(idxTraject))
        # ------------------------
        # The size of figure to 
        # display
        # ------------------------
        subplotWidth    = 4  # inches
        subplotHeight   = 4  # inches

        figWidth    = numCol * subplotWidth
        figHeight   = numRow * subplotHeight

        # ------------------------
        # Create subplot
        # ------------------------
        fig, axes = plt.subplots(numRow, numCol, figsize=(figWidth,figHeight), sharey=False)
        # print("=====================", axes)
        axes = axes.flatten()

        for idxEachHi in range(len(idxTraject.unique())):            # Run for each hi element inside each component
            for idxComp in range(len(inYLabel)):             # Run for each component
            
                axes[idxEachHi].plot(inX[idxTraject == idxEachHi], his[idxTraject == idxEachHi][inYLabel[idxComp]],color=color[idxComp])

            axes[idxEachHi].set_title("T" + str(idxEachHi+1))
            axes[idxEachHi].grid(True)  # Add grid
            if (len(yLim) != 0):
                axes[idxEachHi].set_ylim(yLim) 
        plt.tight_layout()
        plt.show()

In [ ]:
subPlotMultiImgPandas(inData=dfHIAll,numRow=20, numCol=5,color=['green','blue', 'yellow','red'])

In [ ]:
# plot5 = MatPlot() 
# plot5.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"], hiEnv1["Sys"]], inX= timeDraw, color=['green','blue', 'yellow','red'], numRow=20, numCol=5)

In [ ]:
# plot6 = MatPlot( )
# plot6.subPlotMultiImg(inY = [rulEnv1["Sys"]], inX=timeDraw, color=['g'], numRow=20, numCol=5)

In [ ]:
# lookup = {0: -1, 1: 1, 2: 0}
# showMatrix = []
# for idxMulMatrix in hiEnv1["multipliedMatrix"]:
#     signal = [lookup[np.argmax(row)] for row in idxMulMatrix]
#     showMatrix.append(signal)

# plot2 = MatPlot()
# plot2.subPlotMultiImg(inY = [showMatrix], inX=timeDraw, color=['g'], numRow=20, numCol=5, yLim= [-2,2])